# Interactive 3D view of the Knowlytix policy graph

This notebook loads every subject?predicate?object triple from the local GMS store and displays the complete graph in an interactive 3D view. Each triple has its own predicate node, so relations remain distinct.

**Controls:** drag to rotate, scroll to zoom, use the modebar to pan or reset the camera, and hover over nodes for labels and triple details. The full triple table is also included below.

If Plotly is missing from your selected Jupyter environment, run `%pip install -r requirements-visualization.txt`, restart the kernel, and rerun this notebook. The visualization is local and does not make LLM or network requests.


In [1]:
from pathlib import Path
import sys

import networkx as nx
import pandas as pd
import plotly.graph_objects as go

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import loanguard as loan


knowlytix-core v1.4.0 licensed to customer=kalenilesh394@gmail.com tier=developer expires=2027-03-28
By using this software you agree to https://knowlytix.ai/eula (suppress: set KNOWLYTIX_EULA_ACCEPTED=1 or touch C:\Users\nilesh\.knowlytix\eula-accepted).


## Load the GMS graph and list every triple

The project helper loads the store using the geometry settings with which it was built. If no store exists yet, it ingests `loan_policy.md` and creates one.


In [2]:
store = loan.build_or_load_store(rebuild=False, epochs=100)
triples = [tuple(map(str, triple)) for triple in store.doc_graph.triples]
if not triples:
    raise ValueError("The GMS document graph contains no triples.")

triples_df = pd.DataFrame(triples, columns=["subject", "predicate", "object"])
stats = store.stats()
print(
    f"GMS: {stats['entities']} entities, {stats['relations']} relation types, "
    f"{stats['triples']} triples, {stats['enm_entries']} exact numeric values"
)
triples_df


  GMS entities:  52
  GMS relations: 11
  GMS triples:   53
  Store loaded from N:\Solytics\Loan_Approval\loan_store
  Entities:  52
  Relations: 11
  Triples:   53
  ENM:       18
  Documents: 1
[1] loaded existing store from N:\Solytics\Loan_Approval\loan_store
GMS: 52 entities, 11 relation types, 53 triples, 18 exact numeric values


,subject,predicate,object
0,personal loan,has_maximum_amount,25000.0
1,personal loan,has_minimum_credit_score,640.0
2,personal loan,has_maximum_apr,24.99
3,personal loan,has_approval_authority,loan officer
4,personal loan,in_section,loan_product_limits
5,auto loan,has_maximum_amount,60000.0
6,auto loan,has_minimum_credit_score,600.0
7,auto loan,has_maximum_apr,18.99
8,auto loan,has_approval_authority,loan officer
9,auto loan,in_section,loan_product_limits


## Build a 3D force-directed graph

Each triple is represented by a path from its subject through a unique predicate node to its object. This keeps the full triple structure intact, including numeric values stored as literals.


In [3]:
def is_numeric_literal(value: str) -> bool:
    try:
        float(value)
        return True
    except ValueError:
        return False


graph = nx.DiGraph()
node_labels = {}
node_kinds = {}
node_context = {}
for index, (subject, predicate, obj) in enumerate(triples):
    subject_node = ("entity", subject)
    object_kind = "numeric" if is_numeric_literal(obj) else "entity"
    object_node = (object_kind, obj)
    predicate_node = ("predicate", index)
    triple_text = f"{subject} ? {predicate} ? {obj}"

    graph.add_node(subject_node, kind="entity", label=subject)
    graph.add_node(object_node, kind=object_kind, label=obj)
    graph.add_node(predicate_node, kind="predicate", label=predicate)
    graph.add_edge(subject_node, predicate_node)
    graph.add_edge(predicate_node, object_node)

    node_labels[subject_node] = subject
    node_labels[object_node] = obj
    node_labels[predicate_node] = predicate
    node_kinds[subject_node] = "entity"
    node_kinds[object_node] = object_kind
    node_kinds[predicate_node] = "predicate"
    node_context.setdefault(subject_node, set()).add(triple_text)
    node_context.setdefault(object_node, set()).add(triple_text)
    node_context[predicate_node] = {triple_text}

assert sum(kind == "predicate" for kind in node_kinds.values()) == len(triples)
positions = nx.spring_layout(graph, dim=3, seed=27, k=1.15, iterations=300)
print(
    f"3D graph: {len(triples)} triples represented by "
    f"{len(graph.nodes)} nodes and {len(graph.edges)} directed links"
)


3D graph: 53 triples represented by 105 nodes and 106 directed links


In [4]:
# Draw each directed link as a 3D line segment.
edge_x, edge_y, edge_z = [], [], []
for source, target in graph.edges:
    edge_x.extend([positions[source][0], positions[target][0], None])
    edge_y.extend([positions[source][1], positions[target][1], None])
    edge_z.extend([positions[source][2], positions[target][2], None])

traces = [go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode="lines",
    line={"color": "#9aa4b2", "width": 3},
    hoverinfo="skip",
    name="Triple links",
    showlegend=False,
)]
styles = {
    "entity": {"color": "#3182bd", "symbol": "circle", "size": 7, "label": "Entity / text"},
    "numeric": {"color": "#f0a202", "symbol": "square", "size": 8, "label": "Numeric literal"},
    "predicate": {"color": "#8e63a9", "symbol": "diamond", "size": 6, "label": "Predicate"},
}
for kind, style in styles.items():
    nodes = [node for node in graph.nodes if node_kinds[node] == kind]
    if not nodes:
        continue
    customdata = []
    for node in nodes:
        contexts = sorted(node_context.get(node, set()))
        context_text = "<br>".join(contexts[:8])
        if len(contexts) > 8:
            context_text += f"<br>? and {len(contexts) - 8} more triples"
        customdata.append([node_labels[node], style["label"], context_text])
    traces.append(go.Scatter3d(
        x=[positions[node][0] for node in nodes],
        y=[positions[node][1] for node in nodes],
        z=[positions[node][2] for node in nodes],
        mode="markers",
        marker={"size": style["size"], "color": style["color"],
                "symbol": style["symbol"], "line": {"color": "white", "width": 0.7}},
        customdata=customdata,
        hovertemplate=("<b>%{customdata[0]}</b><br>Type: %{customdata[1]}<br>"
                       "Connected triple(s):<br>%{customdata[2]}<extra></extra>"),
        name=style["label"],
    ))

fig = go.Figure(data=traces)
fig.update_layout(
    title=f"Knowlytix GMS policy graph ? {len(triples)} triples",
    height=850,
    margin={"l": 0, "r": 0, "t": 55, "b": 0},
    legend={"orientation": "h", "y": 1.02, "x": 0.02},
    scene={
        "xaxis": {"visible": False}, "yaxis": {"visible": False},
        "zaxis": {"visible": False}, "aspectmode": "data",
        "dragmode": "orbit",
        "camera": {"projection": {"type": "perspective"}},
        "bgcolor": "#f8fafc",
    },
)
fig.show(config={"scrollZoom": True, "displaylogo": False, "responsive": True})


## Graph notes

- Blue circles represent subject or text-value nodes.
- Gold squares represent numeric literal values.
- Purple diamonds are predicate nodes; hover them to see the complete triple.
- The graph is the GMS document graph. Exact numeric values are also retained separately in Knowlytix ENM and can be inspected in notebook 01.
- The force-directed layout is for visual exploration. Node positions do not represent policy meaning or importance.
